# BM25 Foundations

You will build BM25 from scratch in this notebook.

You will learn when keyword retrieval works and when it misses meaning.


## BM25 Idea

You score each document with term frequency, term rarity, and length normalization.

You will inspect each part through code.


In [ ]:
import math
import re
from collections import Counter
from dataclasses import dataclass

## Step 1 Tokenization

You will run a simple tokenizer in the next cell.


In [ ]:
def tokenize(text: str) -> list[str]:
    """Simple tokenizer: lowercase, split on non-alphanumeric, filter short tokens."""
    text = text.lower()
    tokens = re.findall(r'\b\w+\b', text)
    return [t for t in tokens if len(t) > 1]  # Filter single characters

# Test it
sample = "The quick brown fox jumps over the lazy dog."
print(tokenize(sample))

## Step 2 BM25 Class

You will implement indexing, scoring, and search in one class.


In [ ]:
@dataclass
class BM25:
    """BM25 ranking implementation from scratch."""
    k1: float = 1.5  # Term frequency saturation
    b: float = 0.75  # Length normalization
    
    def __post_init__(self):
        self.corpus = []
        self.doc_freqs = []  # Term frequencies per document
        self.doc_lengths = []
        self.avgdl = 0
        self.idf = {}  # IDF scores per term
        self.n_docs = 0
    
    def index(self, documents: list[str]):
        """Index a corpus of documents."""
        self.corpus = documents
        self.n_docs = len(documents)
        
        # Tokenize and compute term frequencies
        self.doc_freqs = []
        self.doc_lengths = []
        doc_containing_term = Counter()  # How many docs contain each term
        
        for doc in documents:
            tokens = tokenize(doc)
            self.doc_lengths.append(len(tokens))
            
            freq = Counter(tokens)
            self.doc_freqs.append(freq)
            
            # Count unique terms in this doc
            for term in set(tokens):
                doc_containing_term[term] += 1
        
        # Compute average document length
        self.avgdl = sum(self.doc_lengths) / self.n_docs if self.n_docs > 0 else 0
        
        # Compute IDF for each term
        self.idf = {}
        for term, n in doc_containing_term.items():
            # BM25 IDF formula
            self.idf[term] = math.log((self.n_docs - n + 0.5) / (n + 0.5) + 1)
        
        print(f"Indexed {self.n_docs} documents")
        print(f"Average document length: {self.avgdl:.1f} tokens")
        print(f"Vocabulary size: {len(self.idf)} terms")
    
    def score(self, query: str, doc_idx: int) -> float:
        """Score a single document against a query."""
        query_terms = tokenize(query)
        doc_freq = self.doc_freqs[doc_idx]
        doc_len = self.doc_lengths[doc_idx]
        
        score = 0.0
        for term in query_terms:
            if term not in self.idf:
                continue  # Term not in corpus
            
            tf = doc_freq.get(term, 0)  # Term frequency in this doc
            idf = self.idf[term]
            
            # BM25 scoring formula
            numerator = tf * (self.k1 + 1)
            denominator = tf + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
            score += idf * (numerator / denominator)
        
        return score
    
    def search(self, query: str, top_k: int = 5) -> list[tuple[int, float, str]]:
        """Search the corpus and return top-k results."""
        scores = [(i, self.score(query, i)) for i in range(self.n_docs)]
        scores.sort(key=lambda x: x[1], reverse=True)
        
        results = []
        for doc_idx, score in scores[:top_k]:
            if score > 0:
                results.append((doc_idx, score, self.corpus[doc_idx][:100] + "..."))
        return results

## Step 3 Test BM25

You will index a toy corpus and inspect ranking behavior.


In [ ]:
# Sample corpus about programming topics
documents = [
    "Python is a high-level programming language known for its simplicity and readability. It supports multiple programming paradigms.",
    "Machine learning is a subset of artificial intelligence that enables systems to learn from data without being explicitly programmed.",
    "Neural networks are computing systems inspired by biological neural networks. They are used in deep learning applications.",
    "Natural language processing (NLP) is a field of AI that focuses on the interaction between computers and human language.",
    "Database systems store and organize data. SQL is a common language for querying relational databases.",
    "Web development involves building websites and web applications using HTML, CSS, JavaScript, and backend technologies.",
    "APIs allow different software systems to communicate with each other. REST and GraphQL are popular API architectures.",
    "Version control systems like Git help developers track changes in code and collaborate effectively on projects.",
    "Cloud computing provides on-demand computing resources over the internet. AWS, Azure, and GCP are major providers.",
    "Cybersecurity involves protecting computer systems and networks from digital attacks and unauthorized access.",
]

# Index the corpus
bm25 = BM25(k1=1.5, b=0.75)
bm25.index(documents)

In [ ]:
# Search for different queries
queries = [
    "machine learning artificial intelligence",
    "programming language",
    "database SQL",
    "neural networks deep learning",
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: '{query}'")
    print(f"{'='*60}")
    
    results = bm25.search(query, top_k=3)
    for rank, (doc_idx, score, snippet) in enumerate(results, 1):
        print(f"\n{rank}. [Score: {score:.3f}]")
        print(f"   {snippet}")

## Inspect IDF

You will see which terms are rare and which terms are common.


In [ ]:
# Sort terms by IDF
sorted_idf = sorted(bm25.idf.items(), key=lambda x: x[1], reverse=True)

print("Highest IDF (rare terms - most discriminative):")
for term, idf in sorted_idf[:10]:
    print(f"  {term}: {idf:.3f}")

print("\nLowest IDF (common terms - least discriminative):")
for term, idf in sorted_idf[-10:]:
    print(f"  {term}: {idf:.3f}")

## Parameter Effects

You will test how k1 and b change ranking outcomes.


In [ ]:
query = "programming language"

# Test different parameter combinations
configs = [
    (1.2, 0.75, "Default (k1=1.2, b=0.75)"),
    (2.0, 0.75, "Higher k1 (k1=2.0, b=0.75)"),
    (1.2, 0.0, "No length norm (k1=1.2, b=0.0)"),
    (1.2, 1.0, "Full length norm (k1=1.2, b=1.0)"),
]

print(f"Query: '{query}'\n")

for k1, b, desc in configs:
    bm25_test = BM25(k1=k1, b=b)
    bm25_test.index(documents)
    
    results = bm25_test.search(query, top_k=3)
    print(f"\n{desc}:")
    for rank, (doc_idx, score, _) in enumerate(results, 1):
        print(f"  {rank}. Doc {doc_idx}: {score:.3f}")

## BM25 Limits

You will see strong exact matching and weak synonym handling.


In [ ]:
# These should find similar documents but won't with BM25
semantic_queries = [
    ("machine learning", "AI that learns from data"),
    ("programming language", "coding syntax"),
    ("web development", "building websites"),
]

print("Semantic gap demonstration:")
print("="*60)

for exact, semantic in semantic_queries:
    print(f"\nExact: '{exact}'")
    exact_results = bm25.search(exact, top_k=1)
    if exact_results:
        print(f"  -> Doc {exact_results[0][0]}, Score: {exact_results[0][1]:.3f}")
    
    print(f"Semantic: '{semantic}'")
    semantic_results = bm25.search(semantic, top_k=1)
    if semantic_results:
        print(f"  -> Doc {semantic_results[0][0]}, Score: {semantic_results[0][1]:.3f}")
    else:
        print(f"  -> No results (terms not in corpus!)")

## Exercises With Starter and Solution

### Exercise 1

You will implement BM25Plus with non negative IDF.


In [ ]:
# Starter code for Exercise 1
class BM25Plus(BM25):
    # TODO implement BM25Plus index with non negative IDF
    def index(self, documents: list[str]):
        pass


In [ ]:
# Solution for Exercise 1
class BM25Plus(BM25):
    # BM25Plus with non negative IDF
    def index(self, documents: list[str]):
        self.corpus = documents
        self.n_docs = len(documents)

        self.doc_freqs = []
        self.doc_lengths = []
        doc_containing_term = Counter()

        for doc in documents:
            tokens = tokenize(doc)
            self.doc_lengths.append(len(tokens))
            freq = Counter(tokens)
            self.doc_freqs.append(freq)
            for term in set(tokens):
                doc_containing_term[term] += 1

        self.avgdl = sum(self.doc_lengths) / self.n_docs if self.n_docs > 0 else 0
        self.idf = {}
        for term, n in doc_containing_term.items():
            raw_idf = math.log((self.n_docs - n + 0.5) / (n + 0.5))
            self.idf[term] = max(0.0, raw_idf)

bm25_plus = BM25Plus(k1=1.5, b=0.75)
bm25_plus.index(documents)
print(bm25_plus.search('programming language', top_k=3))


### Exercise 2

You will add stopword removal to tokenization.


In [ ]:
# Starter code for Exercise 2
STOPWORDS = {'the', 'is', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by'}

def tokenize_with_stopwords(text: str) -> list[str]:
    # TODO implement stopword filtering
    pass


In [ ]:
# Solution for Exercise 2
STOPWORDS = {'the', 'is', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by'}

def tokenize_with_stopwords(text: str) -> list[str]:
    text = text.lower()
    tokens = re.findall(r'\b\w+\b', text)
    return [t for t in tokens if len(t) > 1 and t not in STOPWORDS]

print(tokenize_with_stopwords('The Python language is used for web development and data science'))


### Exercise 3

You will add optional query term weighting to search.


In [ ]:
# Starter code for Exercise 3
def weighted_search(bm25: BM25, query: str, weights: dict = None, top_k: int = 5):
    # TODO apply optional term weights inside BM25 scoring
    pass


In [ ]:
# Solution for Exercise 3
def weighted_search(bm25: BM25, query: str, weights: dict = None, top_k: int = 5):
    if weights is None:
        weights = {}

    query_terms = tokenize(query)
    query_counts = Counter(query_terms)

    scored = []
    for doc_idx in range(bm25.n_docs):
        doc_freq = bm25.doc_freqs[doc_idx]
        doc_len = bm25.doc_lengths[doc_idx]
        score = 0.0

        for term, q_count in query_counts.items():
            if term not in bm25.idf:
                continue

            tf = doc_freq.get(term, 0)
            if tf == 0:
                continue

            term_weight = weights.get(term, 1.0)
            idf = bm25.idf[term]
            numerator = tf * (bm25.k1 + 1)
            denominator = tf + bm25.k1 * (1 - bm25.b + bm25.b * doc_len / bm25.avgdl)
            score += term_weight * q_count * idf * (numerator / denominator)

        scored.append((doc_idx, score, bm25.corpus[doc_idx][:100] + '...'))

    scored.sort(key=lambda x: x[1], reverse=True)
    return [x for x in scored[:top_k] if x[1] > 0]

print(weighted_search(bm25, 'machine learning neural', weights={'neural': 2.0}, top_k=3))


## What You Learned

You implemented BM25 end to end.

You learned weighting, normalization, and BM25 limits.
